# `matrix_function_backward_sum_1` — Interactive Visualization

**Full backward pass** of the scalar loss `L = sum(σ(X @ W))` w.r.t. **X**, with every chain-rule step made explicit:

$$
\underbrace{\frac{dL}{dS} = \mathbf{1}}_{\text{sum rule}}
\quad\longrightarrow\quad
\underbrace{\frac{dL}{dN} = \frac{dL}{dS} \cdot \sigma'(N) = \sigma'(N)}_{\text{through } \sigma}
\quad\longrightarrow\quad
\underbrace{\frac{dL}{dX} = \sigma'(N) \cdot W^T}_{\text{through matmul}}
$$

The key difference from `matrix_function_backward_1`: **dL/dS is always all-ones** because `L = Σ S` treats every element of S equally. That uniform gradient is what makes scalar loss backprop clean.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import ipywidgets as widgets
from IPython.display import display

def sigmoid(x):    return 1 / (1 + np.exp(-x))
def relu(x):       return np.maximum(0, x)
def tanh_(x):      return np.tanh(x)
def leaky_relu(x): return np.where(x > 0, x, 0.01 * x)
def linear(x):     return x
SIGMAS = {"sigmoid": sigmoid, "tanh": tanh_, "relu": relu,
          "leaky_relu": leaky_relu, "linear": linear}

def deriv(func, inp, delta=0.001):
    return (func(inp + delta) - func(inp - delta)) / (2 * delta)

def matrix_function_backward_sum_1(X, W, sigma):
    N    = X @ W
    S    = sigma(N)
    L    = float(np.sum(S))
    dLdS = np.ones_like(S)          # sum rule: every element contributes 1
    dSdN = deriv(sigma, N)
    dLdN = dLdS * dSdN              # = dSdN since dLdS is all 1s
    dNdX = W.T
    dLdX = np.dot(dLdN, dNdX)
    return N, S, L, dLdS, dSdN, dLdN, dNdX, dLdX

print("Setup complete.")

In [ ]:
ONES_COLOR  = "#ea580c"   # orange — highlights the all-ones matrix
BWD_COLOR   = "#dc2626"   # red    — other backward quantities
FWD_COLOR   = "#1d4ed8"   # blue   — forward quantities
GRAD_COLOR  = "#7c3aed"   # purple — final gradient

def heatmap(ax, data, title, title_color="black", cmap="RdBu_r",
            center=None, fixed_vmin=None, fixed_vmax=None):
    if fixed_vmin is not None:
        vmin, vmax = fixed_vmin, fixed_vmax
        norm = None
    else:
        vmax = np.abs(data).max() or 1.0
        vmin = -vmax if center == 0 else data.min()
        norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax) if center == 0 else None
    im = ax.imshow(data, cmap=cmap, norm=norm, vmin=vmin if norm is None else None,
                   vmax=vmax if norm is None else None,
                   aspect="auto", interpolation="nearest")
    plt.colorbar(im, ax=ax, shrink=0.75, pad=0.02)
    r, c = data.shape
    for ri in range(r):
        for ci in range(c):
            v = data[ri, ci]
            mid = (vmax + vmin) / 2 if fixed_vmin is not None else 0
            col = "white" if abs(v - mid) > 0.5 * (vmax - vmin) else "black"
            ax.text(ci, ri, f"{v:.2f}", ha="center", va="center",
                    fontsize=max(5, 9 - max(r, c)), color=col)
    ax.set_title(title, fontsize=10, fontweight="bold", color=title_color, pad=4)
    ax.set_xticks(range(c)); ax.set_yticks(range(r))
    ax.set_xticklabels([f"c{i}" for i in range(c)], fontsize=7)
    ax.set_yticklabels([f"r{i}" for i in range(r)], fontsize=7)
    for sp in ax.spines.values():
        sp.set_edgecolor(title_color); sp.set_linewidth(1.8)

def formula_box(ax, text, fc="#f8fafc", ec="#94a3b8"):
    ax.axis("off")
    ax.text(0.5, 0.5, text, ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.5", fc=fc, ec=ec, lw=1.5))

def draw(batch, in_feat, out_feat, sigma_name, seed):
    sigma = SIGMAS[sigma_name]
    rng   = np.random.default_rng(seed)
    X     = rng.standard_normal((batch, in_feat))
    W     = rng.standard_normal((in_feat, out_feat))

    N, S, L, dLdS, dSdN, dLdN, dNdX, dLdX = matrix_function_backward_sum_1(X, W, sigma)

    # σ' curve
    N_flat = N.flatten()
    pad    = max(2.5, np.abs(N_flat).max() * 0.4)
    xs     = np.linspace(N_flat.min() - pad, N_flat.max() + pad, 500)
    dys    = deriv(sigma, xs)
    dy_pts = deriv(sigma, N_flat)

    # ── Layout: 5 rows × 4 cols ───────────────────────────────────────────────
    # Row 0: X, W
    # Row 1: N, S, L scalar
    # Row 2: dLdS (ONES — highlighted), dSdN, formula: dLdN = dLdS * dSdN
    # Row 3: dLdN (=dSdN), dNdX = W.T, formula, dLdX (gradient)
    # Row 4: σ'(x) curve spanning full width
    fig = plt.figure(figsize=(16, 18))
    fig.suptitle(
        f"matrix_function_backward_sum_1  |  σ={sigma_name}  |  "
        f"X:{X.shape}  W:{W.shape}  →  dL/dX:{dLdX.shape}",
        fontsize=13, fontweight="bold", y=0.995,
    )
    gs = gridspec.GridSpec(5, 4, figure=fig, hspace=0.65, wspace=0.45)

    # ── Row 0: inputs (forward) ───────────────────────────────────────────
    heatmap(fig.add_subplot(gs[0, 0:2]), X, f"X  {X.shape}  (input)",   FWD_COLOR, center=0)
    heatmap(fig.add_subplot(gs[0, 2:4]), W, f"W  {W.shape}  (weights)", FWD_COLOR, center=0)

    # ── Row 1: forward recompute ──────────────────────────────────────────
    heatmap(fig.add_subplot(gs[1, 0:2]), N, f"N = X @ W  {N.shape}  (pre-activation)", FWD_COLOR, center=0)
    heatmap(fig.add_subplot(gs[1, 2:3]), S, f"S = σ(N)  {S.shape}  (post-activation)", FWD_COLOR)

    ax_L = fig.add_subplot(gs[1, 3])
    ax_L.axis("off")
    ax_L.text(0.5, 0.62, "L = Σ S", ha="center", va="center", fontsize=11, color="gray")
    ax_L.text(0.5, 0.30, f"{L:.4f}", ha="center", va="center",
              fontsize=24, fontweight="bold", color=FWD_COLOR,
              bbox=dict(boxstyle="round,pad=0.4", fc="#eff6ff", ec=FWD_COLOR, lw=2))
    ax_L.set_title("Scalar loss L", fontsize=10, fontweight="bold", color=FWD_COLOR)

    # ── Row 2: dLdS (all ones) → dSdN → multiply ─────────────────────────
    # dLdS gets special treatment: fixed vmin=0 vmax=1, orange border
    heatmap(fig.add_subplot(gs[2, 0]), dLdS,
            f"dL/dS = ones_like(S)  {dLdS.shape}\n← sum rule: L=ΣS, so ∂L/∂sᵢ=1",
            ONES_COLOR, cmap="Oranges", fixed_vmin=0.0, fixed_vmax=1.5)

    heatmap(fig.add_subplot(gs[2, 1]), dSdN,
            f"dS/dN = σ'(N)  {dSdN.shape}", BWD_COLOR)

    formula_box(fig.add_subplot(gs[2, 2]),
                "dL/dN =\ndL/dS × dS/dN\n= 1 × σ'(N)\n= σ'(N)",
                fc="#fff7ed", ec=ONES_COLOR)

    # dLdN == dSdN numerically, show side-by-side annotation
    ax_dLdN_preview = fig.add_subplot(gs[2, 3])
    heatmap(ax_dLdN_preview, dLdN,
            f"dL/dN  {dLdN.shape}\n(= dS/dN since dL/dS = 1)",
            BWD_COLOR)

    # ── Row 3: dLdN, dNdX, formula, final gradient ────────────────────────
    heatmap(fig.add_subplot(gs[3, 0]), dLdN,
            f"dL/dN = σ'(N)  {dLdN.shape}", BWD_COLOR)
    heatmap(fig.add_subplot(gs[3, 1]), dNdX,
            f"dN/dX = Wᵀ  {dNdX.shape}", BWD_COLOR, center=0)

    formula_box(fig.add_subplot(gs[3, 2]),
                "dL/dX =\ndL/dN @ dN/dX\n= σ'(N) @ Wᵀ",
                fc="#faf5ff", ec=GRAD_COLOR)

    heatmap(fig.add_subplot(gs[3, 3]), dLdX,
            f"GRADIENT  dL/dX  {dLdX.shape}", GRAD_COLOR, center=0)

    # ── Row 4: σ'(x) curve with N values ─────────────────────────────────
    ax_ds = fig.add_subplot(gs[4, 0:4])
    ax_ds.plot(xs, dys, color=BWD_COLOR, lw=2.5, label="σ'(x)")
    ax_ds.axhline(0, color="#9ca3af", lw=0.8, ls="--")
    ax_ds.scatter(N_flat, dy_pts, color=ONES_COLOR, s=60, zorder=5,
                  label=f"σ'(N) = dL/dN values  (n={len(N_flat)})")
    for xv, yv in zip(N_flat, dy_pts):
        ax_ds.vlines(xv, 0, yv, color=ONES_COLOR, lw=0.7, alpha=0.4, ls="--")
    ax_ds.set_title("σ'(x) — derivative  (these values fill dL/dN, then get multiplied by Wᵀ to produce dL/dX)",
                    fontsize=10, fontweight="bold", color=BWD_COLOR)
    ax_ds.set_xlabel("x  (pre-activation N values)", fontsize=9)
    ax_ds.set_ylabel("σ'(x)", fontsize=9)
    ax_ds.legend(fontsize=9); ax_ds.grid(True, alpha=0.3)
    for sp in ax_ds.spines.values():
        sp.set_edgecolor(BWD_COLOR); sp.set_linewidth(1.4)

    plt.show()
    print(f"L={L:.6f}  |  dLdX shape:{dLdX.shape}  min:{dLdX.min():.4f}  max:{dLdX.max():.4f}")

print("Draw function ready.")

In [ ]:
style  = {"description_width": "155px"}
layout = widgets.Layout(width="430px")

w_batch    = widgets.IntSlider(value=3, min=1, max=8, description="batch (rows X)",            style=style, layout=layout)
w_in_feat  = widgets.IntSlider(value=4, min=1, max=8, description="in_feat (cols X = rows W)", style=style, layout=layout)
w_out_feat = widgets.IntSlider(value=2, min=1, max=8, description="out_feat (cols W)",         style=style, layout=layout)
w_sigma    = widgets.Dropdown(options=list(SIGMAS.keys()), value="sigmoid",
                              description="σ (activation)", style=style)
w_seed     = widgets.IntSlider(value=42, min=0, max=999, description="random seed",            style=style, layout=layout)

ui = widgets.VBox([
    widgets.HTML("<b>Matrix dimensions</b>"),
    w_batch, w_in_feat, w_out_feat,
    widgets.HTML("<b>Activation &amp; randomness</b>"),
    w_sigma, w_seed,
])

out = widgets.interactive_output(
    draw,
    {"batch": w_batch, "in_feat": w_in_feat, "out_feat": w_out_feat,
     "sigma_name": w_sigma, "seed": w_seed},
)

display(ui, out)